Notebook 02 - Database Schema

Cell 1  Database Connection

Cell 2  Create raw_bhavcopy

Cell 3  Create india_vix

Cell 4  Create risk_free_rate

Cell 5  Create expiry_calendar

Cell 6  Create lot_size_history

Cell 7  Create master_dataset

Cell 8  Create etl_file_log   ← here

Cell 9  Create etl_job_log

Cell 10 Verify Tables

In [3]:
#Bootstrap
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
from sqlalchemy import text

from config.settings import *
from config.database import engine

In [5]:
print("="*60)
print("DATABASE CONNECTION")
print("="*60)

with engine.connect() as conn:

    version = conn.execute(text("SELECT VERSION()")).scalar()

print("Connection Status : SUCCESS")
print("MariaDB Version   :", version)

DATABASE CONNECTION
Connection Status : SUCCESS
MariaDB Version   : 12.1.2-MariaDB


In [4]:
#Create Schema

create_database_sql = """
CREATE DATABASE IF NOT EXISTS option_writing;
"""

with engine.begin() as conn:
    conn.execute(text(create_database_sql))

print("Database Verified.")

Database Verified.


In [18]:
create_raw_bhavcopy = """
CREATE TABLE IF NOT EXISTS raw_bhavcopy
(
    id BIGINT AUTO_INCREMENT PRIMARY KEY,

    instrument         VARCHAR(10) NOT NULL,
    symbol             VARCHAR(30) NOT NULL,
    expiry_date        DATE NOT NULL,
    strike_price       DECIMAL(12,2) NOT NULL,
    option_type        VARCHAR(2) NOT NULL,

    open_price         DECIMAL(12,2),
    high_price         DECIMAL(12,2),
    low_price          DECIMAL(12,2),
    close_price        DECIMAL(12,2),
    settle_price       DECIMAL(12,2),

    contracts          BIGINT,
    value_in_lakh      DECIMAL(20,4),

    open_interest      BIGINT,
    change_in_oi       BIGINT,

    trading_date       DATE NOT NULL,

    created_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT uk_option_contract UNIQUE
    (
        instrument,
        symbol,
        expiry_date,
        strike_price,
        option_type,
        trading_date
    )
);
"""

with engine.begin() as conn:
    conn.execute(text(create_raw_bhavcopy))

print("✓ raw_bhavcopy recreated.")

✓ raw_bhavcopy recreated.


In [8]:
create_vix = """

CREATE TABLE IF NOT EXISTS india_vix
(
    trading_date      DATE PRIMARY KEY,

    open_price        DECIMAL(8,4) NOT NULL,
    high_price        DECIMAL(8,4) NOT NULL,
    low_price         DECIMAL(8,4) NOT NULL,
    close_price       DECIMAL(8,4) NOT NULL,

    previous_close    DECIMAL(8,4),
    change_points     DECIMAL(8,4),
    change_percent    DECIMAL(8,4),

    created_at        TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

"""

with engine.begin() as conn:
    conn.execute(text(create_vix))

print("india_vix created.")

india_vix created.


In [9]:
create_rf = """

CREATE TABLE IF NOT EXISTS risk_free_rate
(
    auction_date              DATE PRIMARY KEY,
    issue_date                DATE NOT NULL,

    notified_amount           DECIMAL(15,2),

    cutoff_price              DECIMAL(10,4),

    cutoff_yield              DECIMAL(8,4) NOT NULL,

    weighted_average_price    DECIMAL(10,4),

    weighted_average_yield    DECIMAL(8,4),

    created_at                TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

with engine.begin() as conn:
    conn.execute(text(create_rf))

print("risk_free_rate created.")

risk_free_rate created.


In [10]:
#create expiry calendar

create_expiry = """

CREATE TABLE IF NOT EXISTS expiry_calendar
(

    expiry DATE PRIMARY KEY,

    expiry_type VARCHAR(20),

    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP

);

"""

with engine.begin() as conn:
    conn.execute(text(create_expiry))

print("expiry_calendar created.")

expiry_calendar created.


In [11]:
#Create Master Dataset
create_master = """

CREATE TABLE IF NOT EXISTS master_dataset
(

    id BIGINT AUTO_INCREMENT PRIMARY KEY,

    trading_date DATE,

    symbol VARCHAR(30),

    expiry DATE,

    strike DECIMAL(12,2),

    option_type VARCHAR(5),

    days_to_expiry INT,

    underlying_close DECIMAL(12,2),

    option_close DECIMAL(12,2),

    implied_volatility DECIMAL(10,4),

    open_interest BIGINT,

    change_in_oi BIGINT,

    india_vix DECIMAL(10,4),

    risk_free_rate DECIMAL(8,4),

    iv_rank DECIMAL(8,4),

    iv_percentile DECIMAL(8,4),

    realized_volatility DECIMAL(8,4),

    target_expired_worthless TINYINT,

    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP

);

"""

with engine.begin() as conn:
    conn.execute(text(create_master))

print("master_dataset created.")

master_dataset created.


In [12]:
#Create Index

indexes = [

"""
CREATE INDEX idx_trade_date
ON raw_bhavcopy(trading_date);
""",

"""
CREATE INDEX idx_symbol
ON raw_bhavcopy(symbol);
""",

"""
CREATE INDEX idx_expiry
ON raw_bhavcopy(expiry);
""",

"""
CREATE INDEX idx_symbol_expiry
ON raw_bhavcopy(symbol,expiry);
""",

"""
CREATE INDEX idx_symbol_expiry_strike
ON raw_bhavcopy(symbol,expiry,strike);
"""

]

with engine.begin() as conn:

    for idx in indexes:
        try:
            conn.execute(text(idx))
        except:
            pass

print("Indexes Created.")

Indexes Created.


In [13]:
#Checking tables
query = """
SHOW TABLES;
"""

with engine.connect() as conn:

    tables = conn.execute(text(query)).fetchall()

for table in tables:
    print(table[0])

expiry_calendar
india_vix
master_dataset
raw_bhavcopy
risk_free_rate


In [15]:
create_indexes = """

CREATE INDEX idx_trade_date
ON raw_bhavcopy(trading_date);

CREATE INDEX idx_symbol
ON raw_bhavcopy(symbol);

CREATE INDEX idx_expiry
ON raw_bhavcopy(expiry_date);

CREATE INDEX idx_instrument
ON raw_bhavcopy(instrument);

CREATE INDEX idx_symbol_expiry
ON raw_bhavcopy(symbol, expiry_date);

CREATE INDEX idx_symbol_expiry_strike
ON raw_bhavcopy(symbol, expiry_date, strike_price);

CREATE INDEX idx_option_lookup
ON raw_bhavcopy(
    symbol,
    expiry_date,
    strike_price,
    option_type
);

"""

for query in create_indexes.split(";"):
    if query.strip():
        try:
            with engine.begin() as conn:
                conn.execute(text(query))
        except Exception:
            pass

print("✓ Indexes created successfully.")

✓ Indexes created successfully.


In [25]:
from sqlalchemy import text

create_etl_file_log = """
DROP TABLE IF EXISTS etl_file_log;

CREATE TABLE etl_file_log
(
    id BIGINT AUTO_INCREMENT PRIMARY KEY,

    file_name VARCHAR(255) NOT NULL,

    file_size BIGINT NOT NULL,

    file_hash CHAR(64) NOT NULL,

    table_name VARCHAR(100) NOT NULL,

    records_loaded INT DEFAULT 0,

    status ENUM('SUCCESS','FAILED','SKIPPED') NOT NULL,

    remarks VARCHAR(1000),

    load_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

    UNIQUE KEY uk_file_hash (table_name, file_hash),

    INDEX idx_file_name (file_name),

    INDEX idx_status (status)
);
"""

with engine.begin() as conn:

    for stmt in create_etl_file_log.split(";"):
        if stmt.strip():
            conn.execute(text(stmt))

print("✓ etl_file_log recreated successfully.")

✓ etl_file_log recreated successfully.


In [20]:
#Creating etl_job_log

from sqlalchemy import text

create_job_log = """
CREATE TABLE IF NOT EXISTS etl_job_log
(
    job_id BIGINT AUTO_INCREMENT PRIMARY KEY,

    job_name VARCHAR(100) NOT NULL,

    start_time DATETIME NOT NULL,

    end_time DATETIME,

    total_files INT DEFAULT 0,

    successful_files INT DEFAULT 0,

    failed_files INT DEFAULT 0,

    total_records BIGINT DEFAULT 0,

    status VARCHAR(20),

    remarks VARCHAR(1000),

    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

with engine.begin() as conn:
    conn.execute(text(create_job_log))

print("✓ etl_job_log table created.")

✓ etl_job_log table created.


In [21]:
#Verifying

from sqlalchemy import text

with engine.connect() as conn:
    tables = conn.execute(text("SHOW TABLES")).fetchall()

for table in tables:
    print(table[0])

etl_file_log
etl_job_log
expiry_calendar
india_vix
master_dataset
raw_bhavcopy
risk_free_rate


In [22]:
from datetime import datetime
from sqlalchemy import text

with engine.begin() as conn:

    result = conn.execute(
        text("""
        INSERT INTO etl_job_log
        (
            job_name,
            start_time,
            status
        )
        VALUES
        (
            :job_name,
            :start_time,
            :status
        )
        """),
        {
            "job_name": "Bhavcopy ETL",
            "start_time": datetime.now(),
            "status": "RUNNING"
        }
    )

    job_id = result.lastrowid

print(f"Job Started: {job_id}")

Job Started: 1


In [23]:
from datetime import datetime
from sqlalchemy import text

with engine.begin() as conn:

    conn.execute(
        text("""
        UPDATE etl_job_log
        SET
            end_time = :end_time,
            total_files = :total_files,
            successful_files = :successful_files,
            failed_files = :failed_files,
            total_records = :total_records,
            status = :status
        WHERE job_id = :job_id
        """),
        {
            "end_time": datetime.now(),
            "total_files": 350,
            "successful_files": 349,
            "failed_files": 1,
            "total_records": 6825441,
            "status": "SUCCESS",
            "job_id": job_id
        }
    )

print("Job Completed.")

Job Completed.
